# P2 robustness analysis (post-primary, no model inference)

This notebook reads the hash-verified P1-SHAPE arrays already stored in Google Drive. It computes the owner-approved P2 metric robustness outputs without loading Chronos-2 or TimesFM-3 and without changing the registered P1 decision. CPU runtime is sufficient.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import json
from pathlib import Path
import subprocess
import sys

REPO = Path('/content/tsfm-covariate-faithfulness')
REPO_URL = 'https://github.com/FlyMe2star/tsfm-covariate-faithfulness.git'
if REPO.exists():
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', 'main'], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', 'main'], check=True)
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', 'main'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', 'main', REPO_URL, str(REPO)], check=True)
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO / 'requirements/colab-base.txt')],
    check=True,
)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)
src_path = str(REPO / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)
commit = subprocess.check_output(
    ['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True
).strip()
print('Repository commit:', commit)


In [ ]:
from covfaith.p1_shape import scientific_code_hash as primary_scientific_code_hash
from covfaith_supp.robustness import analyze_p2_robustness, verify_p2_freeze

EXPECTED_CONFIG_HASH = '7f7c27bc37ffc57ce3b2455877dab2a72d7d0d3255484042377cdaf90c27a284'
EXPECTED_SUPPLEMENT_CODE_HASH = '7178cf9bfa72d63a261765fdad333a30abd34ee0014c8bb082d160e6e7eebbe4'
EXPECTED_PRIMARY_CODE_HASH = 'fc1cf6d73a59a4200deaa964f341a17b72d8fae75fa40b1086ed1c8c714f5925'
INPUT_ROOT = Path('/content/drive/MyDrive/tsfm-covariate-faithfulness/p1_shape_v1')
OUTPUT_ROOT = Path('/content/drive/MyDrive/tsfm-covariate-faithfulness/p2_robustness_v1')

assert INPUT_ROOT.exists(), f'Missing archived P1 root: {INPUT_ROOT}'
hashes = verify_p2_freeze(REPO)
assert hashes['config_hash'] == EXPECTED_CONFIG_HASH
assert hashes['scientific_code_sha256'] == EXPECTED_SUPPLEMENT_CODE_HASH
assert primary_scientific_code_hash(REPO) == EXPECTED_PRIMARY_CODE_HASH
print(json.dumps(hashes, indent=2))
print('Verified read-only P1 root:', INPUT_ROOT)
print('Durable P2 output root:', OUTPUT_ROOT)


In [ ]:
report = analyze_p2_robustness(REPO, INPUT_ROOT, OUTPUT_ROOT)
print(json.dumps(report, indent=2, ensure_ascii=False))
print('P2 analysis complete. No model inference was performed.')


In [ ]:
import pandas as pd

for name in [
    'monotone_cell_summary.csv',
    'wql_cell_summary.csv',
    'metric_fixture_comparison.csv',
    'threshold_sensitivity.csv',
]:
    print('\n###', name)
    display(pd.read_csv(OUTPUT_ROOT / name))

representatives = json.loads(
    (OUTPUT_ROOT / 'representative_responses.json').read_text(encoding='utf-8')
)
print('Representative series:', [item['cell'] + '/' + item['series_id'] for item in representatives])


In [ ]:
import shutil
from google.colab import files

archive = shutil.make_archive('/content/p2_robustness_v1', 'zip', OUTPUT_ROOT)
print('Download bundle:', archive)
files.download(archive)
